In [1]:
import os
import sys
import open3d as o3d

def read_with_open3d(file_path):
    pcd = o3d.io.read_point_cloud(file_path)
    return pcd

pcd = read_with_open3d('/home/ajoy/saumya/cv/ModelNet-10/train/chair/chair_0002.ply')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
import numpy as np
import torch
pts = np.asarray(pcd.points)

inp = torch.tensor(pts.astype(np.float32)).unsqueeze(0).transpose(1, 2)
pts.shape, inp.shape

((5556, 3), torch.Size([1, 3, 5556]))

In [5]:
cls_map = {}
folders = os.listdir('/home/ajoy/saumya/cv/ModelNet-10/train')
for i, folder in enumerate(folders):
    cls_map[folder] = i

In [6]:
cls_map

{'sofa': 0,
 'desk': 1,
 'monitor': 2,
 'chair': 3,
 'table': 4,
 'night_stand': 5,
 'bathtub': 6,
 'bed': 7,
 'dresser': 8,
 'toilet': 9}

In [7]:
from torch.utils.data import Dataset, DataLoader
import open3d as o3d

class modelnet:
    def __init__(self, mode='train', transforms=None, sample_points_count=2048):
        par = 'ModelNet-10'
        folders = os.listdir(os.path.join(par, mode))

        self.sample_points_count = sample_points_count
        self.images = []
        self.labels = []
        for i, folder in enumerate(folders):
            files = os.listdir(f'/home/ajoy/saumya/cv/ModelNet-10/{mode}/{folder}')
            for file in files:
                pcd = o3d.io.read_point_cloud(f'/home/ajoy/saumya/cv/ModelNet-10/{mode}/{folder}/{file}')
                pts = np.asarray(pcd.points)
                pts = pts - pts.mean(axis=0, keepdims=True)
                scale = np.max(np.linalg.norm(pts, axis=1))
                if scale > 0:
                    pts = pts / scale
                inp = torch.tensor(pts.astype(np.float32)).transpose(0, 1)

                self.images.append(inp)
                self.labels.append(cls_map[folder])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        img = self.images[i]
        sampled_indices = np.random.choice(img.shape[-1], self.sample_points_count, replace=(self.sample_points_count>img.shape[-1]))
        img = img[:, sampled_indices]
        return img, self.labels[i]

data = modelnet()

In [8]:
last = []
for i in data.images:
    last.append(i.shape[-1])

In [9]:
train = DataLoader(data, batch_size=16, shuffle=True)
for i in train:
    print(i[0].shape, i[1].shape)

torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size([16, 3, 2048]) torch.Size([16])
torch.Size(

In [10]:
import plotly.graph_objects as go
fig = go.Figure(data=[go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers')])
fig.show()

In [11]:
import torch
from torch import nn

# inp = torch.concat([inp, inp], dim=-1)
print(inp.shape)
out = nn.Conv1d(3, 1024, kernel_size=1)(inp)
print(out.shape)
out = nn.AdaptiveAvgPool1d(1)(out)
out.shape

torch.Size([1, 3, 5556])
torch.Size([1, 1024, 5556])


torch.Size([1, 1024, 1])

In [12]:
import torch.nn.functional as F
import torch.optim as optim

class PointNet(nn.Module):
    def __init__(self):
        super(PointNet, self).__init__()

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 256, 1)
        self.conv4 = nn.Conv1d(256, 512, 1)
        self.conv5 = nn.Conv1d(512, 1024, 1)
        self.convs = nn.Sequential(self.conv1, self.conv2, self.conv3, self.conv4, self.conv5)

        # classification head
        self.cls1 = nn.Linear(1024, 512)
        self.cls2 = nn.Linear(512, 64)
        self.cls3 = nn.Linear(64, 10)
        self.cls = nn.Sequential(self.cls1, self.cls2)

    
    def forward(self, x):
        for i in self.convs:
            x = i(x)
            x = F.relu(x)

        x = torch.max(x, 2).values

        for i in self.cls:
            x = i(x)
            x = F.relu(x)
        
        x = self.cls3(x)
        # x = F.softmax(x, dim=-1)
        return x

pn = PointNet().to("cuda")
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(pn.parameters(), lr=1e-3)
epochs = 10
losses = []

for i in range(epochs):
    pn.train()
    epoch_loss = 0
    for (img, tgs) in train:
        opt.zero_grad()
        img = img.to('cuda')
        tgs = tgs.to('cuda')


        out = pn(img)

        loss = loss_fn(out, tgs)
        loss.backward()

        opt.step()
        epoch_loss += loss
    
    print(f"Loss at epoch {i+1}: {epoch_loss/len(train)}")
    losses.append(epoch_loss/len(train))


Loss at epoch 1: 1.555309534072876
Loss at epoch 2: 0.6787651181221008
Loss at epoch 3: 0.5656954646110535
Loss at epoch 4: 0.5086442828178406
Loss at epoch 5: 0.43742409348487854
Loss at epoch 6: 0.40689077973365784
Loss at epoch 7: 0.3611901104450226
Loss at epoch 8: 0.36250796914100647
Loss at epoch 9: 0.3252109885215759
Loss at epoch 10: 0.30931153893470764


In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class PointNet(nn.Module):
    def __init__(self):
        super(PointNet, self).__init__()

        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 256, 1)
        self.conv4 = nn.Conv1d(256, 512, 1)
        self.conv5 = nn.Conv1d(512, 1024, 1)

        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(256)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(1024)

        # classification head
        self.cls1 = nn.Linear(1024, 64)
        self.cls2 = nn.Linear(64, 10)

        self.bn6 = nn.BatchNorm1d(64)

    def forward(self, x, critical=False):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        if critical:
            return x

        x = torch.max(x, 2).values

        x = F.relu(self.bn6(self.cls1(x)))
        x = self.cls2(x)

        return x


pn = PointNet().to("cuda")
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(pn.parameters(), lr=1e-3)
epochs = 10
losses = []

for i in range(epochs):
    pn.train()
    epoch_loss = 0.0

    for img, tgs in train:
        opt.zero_grad()
        img = img.to("cuda")
        tgs = tgs.to("cuda")

        out = pn(img)
        loss = loss_fn(out, tgs)
        loss.backward()
        opt.step()

        epoch_loss += loss.item()

    print(f"Loss at epoch {i+1}: {epoch_loss / len(train):.4f}")
    losses.append(epoch_loss/len(train))

Loss at epoch 1: 0.5624
Loss at epoch 2: 0.2692
Loss at epoch 3: 0.2067
Loss at epoch 4: 0.1931
Loss at epoch 5: 0.1742
Loss at epoch 6: 0.1735
Loss at epoch 7: 0.1602
Loss at epoch 8: 0.1513
Loss at epoch 9: 0.1310
Loss at epoch 10: 0.1167


In [14]:
test_data = modelnet(mode='test')
test_loader = DataLoader(test_data, batch_size=1, shuffle=True)

c, t = 0, 0
pn.eval()
for img, tgs in test_loader:
    img = img.to("cuda")
    tgs = tgs.to("cuda").long()

    out = pn(img)
    # print(torch.argmax(out))
    if torch.argmax(out) == tgs:
        c += 1

c/len(test_data.images)

0.8744493392070485

In [15]:
## change with permuation
test_data = modelnet(mode='test')
test_loader = DataLoader(test_data, batch_size=1, shuffle=True)

c, t = 0, 0
pn.eval()
for img, tgs in test_loader:
    img = img.to("cuda")
    tgs = tgs.to("cuda").long()

    out_np = pn(img)

    a = np.arange(0, 2048)
    np.random.shuffle(a)
    img = img[:, :, a]
    out_p = pn(img)
    if torch.argmax(out_p) != torch.argmax(out_np):
        c += 1

c/len(test_data.images)

0.0

In [16]:
img = data.images[0]
map = pn(img.unsqueeze(0).to("cuda"), critical=True)

In [17]:
critical_points = torch.argmax(map, 2)[0].cpu().numpy()

In [18]:
pts = img.permute(1, 0).cpu().numpy()

In [19]:
import plotly.graph_objects as go
fig = go.Figure(data=[go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode='markers')])
fig.show()

In [20]:
import plotly.graph_objects as go
fig = go.Figure(data=[go.Scatter3d(x=pts[critical_points,0], y=pts[critical_points,1], z=pts[critical_points,2], mode='markers')])
fig.show()

In [21]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pn.eval()

# choose 5 samples from the test set
sample_ids = [0, 1, 2, 3, 4]   # or use random.sample(range(len(data)), 5)

fig = make_subplots(
    rows=5,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}] for _ in range(5)],
    subplot_titles=[
        title
        for i in sample_ids
        for title in (f"Sample {i} - Full Point Cloud", f"Sample {i} - Critical Points Overlay")
    ],
    horizontal_spacing=0.03,
    vertical_spacing=0.04
)

with torch.no_grad():
    for row, idx in enumerate(sample_ids, start=1):
        img = data.images[idx]   # shape expected: [3, N]
        
        # Get critical point map
        crit_map = pn(img.unsqueeze(0).to("cuda"), critical=True)
        
        # argmax over points dimension -> one critical point index per feature channel
        critical_points = torch.argmax(crit_map, dim=2)[0].cpu().numpy()
        
        # remove duplicates so only unique critical points are shown
        critical_points = np.unique(critical_points)

        # point cloud as [N, 3]
        pts = img.permute(1, 0).cpu().numpy()

        # -------- Left: full point cloud --------
        fig.add_trace(
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode='markers',
                marker=dict(size=2, opacity=0.8),
                showlegend=False
            ),
            row=row, col=1
        )

        # -------- Right: faint full cloud --------
        fig.add_trace(
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode='markers',
                marker=dict(size=2, opacity=0.12, color='gray'),
                showlegend=False
            ),
            row=row, col=2
        )

        # -------- Right: critical points overlay --------
        fig.add_trace(
            go.Scatter3d(
                x=pts[critical_points, 0],
                y=pts[critical_points, 1],
                z=pts[critical_points, 2],
                mode='markers',
                marker=dict(size=4, color='red', opacity=1.0),
                showlegend=False
            ),
            row=row, col=2
        )

# Make all scenes look clean and consistent
for r in range(1, 6):
    for c in range(1, 3):
        scene_id = (r - 1) * 2 + c
        fig.update_layout(**{
            f"scene{scene_id}": dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data"
            )
        })

fig.update_layout(
    height=2200,
    width=1000,
    title="PointNet Critical Points Visualization",
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

In [22]:
test_data = modelnet(mode='test')
test_loader = DataLoader(test_data, batch_size=1, shuffle=True)

c, t = 0, 0
pn.eval()
for i, (img, tgs) in enumerate(test_loader):
    img = img.to("cuda:0")
    tgs = tgs.to("cuda:0").long()

    out = pn(img, critical=True)
    critical_points = torch.argmax(out, dim=2)[0]
    out = pn(img[:, :, critical_points])
    if torch.argmax(out) == tgs:
        c += 1
    
    if i == 100:
        break

c/100

0.87

Accuracy does not drop because of the global max pooling. Also because we are testing now, so the criticial points were the only ones that were going to gert selected for those channels./ 